# Notebook 7: Complete System (RAG + TAG)## Ultimate Chatbot:- PDF document search (RAG)- Genre classification (TAG)- Web search- All tools combined!

In [ ]:
%pip install langchain langchain-community langgraph python-dotenv openpyxl pandas langchain-openai google-search-results pypdf faiss-cpu --quiet

In [ ]:
from langchain_core.tools import toolfrom langchain_community.utilities import GoogleSerperAPIWrapperfrom langgraph.graph import StateGraph, START, END, MessagesStatefrom langgraph.prebuilt import ToolNodefrom langgraph.checkpoint.memory import MemorySaverfrom langchain_core.messages import HumanMessage, SystemMessagefrom typing import Literalfrom ai_course_utils import (    load_llm_from_env,    load_use_case_config,    load_taxonomy_from_excel,    load_pdf_for_rag)

In [ ]:
use_case_file = 'use_case_Movie_Recommendation.xlsx'config = load_use_case_config(use_case_file)system_prompt = config.get('agent_prompt')

## Load All Data Files**USER INPUT**: Provide all file paths

In [ ]:
# ============================================================================# USER INPUT: Specify all file paths# ============================================================================taxonomy_file = 'Movie_Recommendation_Taxonomy_File.xlsx'pdf_file = 'The_98th_Academy_Awards___2026.pdf'# Load taxonomytaxonomy = load_taxonomy_from_excel(taxonomy_file)# Load PDFdocuments, vectorstore, retriever = load_pdf_for_rag(pdf_file)print(f'\n✅ All data loaded!')print(f'  Genres: {len(taxonomy)}')print(f'  PDF Pages: {len(documents)}')

## Define All Three Tools

In [ ]:
@tooldef search_web(query: str) -> str:    """Search web for current movie information."""    search = GoogleSerperAPIWrapper()    return search.run(query)@tooldef search_documents(query: str) -> str:    """Search the Oscars 2026 PDF document."""    docs = retriever.invoke(query)    return '\n\n'.join([doc.page_content for doc in docs])@tooldef classify_genre(query: str) -> dict:    """Classify query into movie genre using taxonomy."""    query_lower = query.lower()    for genre_name, info in taxonomy.items():        included = info.get('included', '').lower()        keywords = [k.strip() for k in included.split(',') if k.strip()]        if any(kw in query_lower for kw in keywords):            return {                'genre': genre_name,                'description': info.get('description', ''),                'guidelines': info.get('guidelines', '')            }    return {'genre': 'General'}tools = [classify_genre, search_documents, search_web]print(f'✓ All 3 tools ready: {[t.name for t in tools]}')

## Enhanced System Prompt

In [ ]:
enhanced_prompt = """You are an expert movie assistant with three powerful capabilities:1. **classify_genre**: Understand user preferences via 10 genre categories2. **search_documents**: Search 2026 Oscars nominations document3. **search_web**: Find current movie informationWORKFLOW:- First classify queries to understand preferences- Use documents for Oscar-related questions- Use web search for current/general info- Combine sources for comprehensive answersBe helpful, accurate, and cite sources."""print('✓ Enhanced prompt configured')

In [ ]:
llm = load_llm_from_env()llm_with_tools = llm.bind_tools(tools)

In [ ]:
def assistant(state):    return {'messages': [llm_with_tools.invoke([SystemMessage(content=enhanced_prompt)] + state['messages'])]}def should_continue(state):    return 'tools' if state['messages'][-1].tool_calls else '__end__'graph_builder = StateGraph(MessagesState)graph_builder.add_node('assistant', assistant)graph_builder.add_node('tools', ToolNode(tools))graph_builder.add_edge(START, 'assistant')graph_builder.add_conditional_edges('assistant', should_continue)graph_builder.add_edge('tools', 'assistant')graph = graph_builder.compile(checkpointer=MemorySaver())print('✓ Complete system built')

In [ ]:
def chat(msg, tid='default', verbose=False):    if verbose:        print(f'\nProcessing: {msg}')    for event in graph.stream({'messages': [HumanMessage(content=msg)]}, {'configurable': {'thread_id': tid}}, stream_mode='values'):        pass    return event['messages'][-1].contentprint('🎬 Complete RAG+TAG system ready!')

## Test Complete System

In [ ]:
# Complex query using multiple capabilitiesquery = 'I love mythology mixed with sci-fi. Are there any Oscar nominees like that?'print(f'User: {query}\n')response = chat(query, verbose=True)print(f'\nBot: {response}')

In [ ]:
# Another complex queryquery2 = 'Who won Best Picture? Recommend similar futuristic noir films'print(f'\nUser: {query2}\n')response2 = chat(query2, verbose=True)print(f'\nBot: {response2}')

## Summary### Complete System Features:✅ Genre classification (10 custom genres)  ✅ PDF document search (Oscars 2026)  ✅ Web search (current information)  ✅ Multi-tool orchestration  ✅ Source attribution  ### All 7 Notebooks Complete!**Progression**:1. Simple chatbot2. + Web search3. + URL fetching4. Data analysis5. + RAG (PDF documents)6. + TAG (Taxonomy)7. RAG + TAG combined ← You are here!### Ready for Streamlit:This architecture is perfect for a Streamlit app where users can:- Upload their own use case files- Upload custom taxonomies- Upload PDF documents- Select solution patterns- Chat with AI### Congratulations!You now have a complete, production-ready, model-agnostic AI system!